In [ ]:
# In[1]:

import pandas as pd
import pytz

# Reuse variable names across steps for efficiency
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Ensure numeric 'value' columns where applicable
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) metric.csv summary: group by (cmdb_id, kpi_name) -> count and global P95
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count=('value', 'count'),
    p95_value=('value', lambda s: float(s.quantile(0.95)))
)
metric_summary = metric_grp.sort_values('count', ascending=False).head(20)

# 2) trace.csv summary: group by (cmdb_id, trace_name) -> count and global P95
trace_grp = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count=('value', 'count'),
    p95_value=('value', lambda s: float(s.quantile(0.95)))
)
trace_summary = trace_grp.sort_values('count', ascending=False).head(20)

# 3) log.csv summary: group by (cmdb_id, log_name) -> count and global P95
log_grp = df_log.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    count=('value', 'count'),
    p95_value=('value', lambda s: float(s.quantile(0.95)))
)
log_summary = log_grp.sort_values('count', ascending=False).head(20)

# 4) error_logs.csv: per-cmdb_id count, earliest and latest timestamp (converted to UTC+8)
if not df_error.empty:
    # Convert timestamp to timezone-aware datetimes (assume epoch seconds UTC then convert to Asia/Shanghai)
    ts = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df_error = df_error.copy()
    df_error['ts_dt'] = ts

    error_agg = df_error.groupby('cmdb_id', as_index=False).agg(
        error_count=('message', 'count'),
        earliest_ts=('ts_dt', 'min'),
        latest_ts=('ts_dt', 'max')
    )

    # Format timestamps as ISO strings (compact)
    error_agg['earliest_ts'] = error_agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_agg['latest_ts'] = error_agg['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    # Keep it compact and limit to 20 rows
    error_summary = error_agg.sort_values('error_count', ascending=False).head(20)

    # Prepare up to 10 sample messages: choose latest message per cmdb_id for top cmdb_ids by count, limit 10
    top_cmdbs = error_summary['cmdb_id'].head(10).tolist()
    # For those cmdb_ids, pick the latest message row
    df_latest_msgs = df_error[df_error['cmdb_id'].isin(top_cmdbs)].sort_values(['cmdb_id', 'ts_dt'], ascending=[True, False])
    # Drop duplicates keep latest per cmdb_id
    error_samples = df_latest_msgs.drop_duplicates(subset=['cmdb_id'], keep='first')[['cmdb_id', 'ts_dt', 'message']].head(10).copy()
    # Format timestamp
    error_samples['ts_dt'] = error_samples['ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    error_summary = pd.DataFrame(columns=['cmdb_id', 'error_count', 'earliest_ts', 'latest_ts'])
    error_samples = pd.DataFrame(columns=['cmdb_id', 'ts_dt', 'message'])

# Display compact summaries (variables returned)
metric_summary, trace_summary, log_summary, error_summary, error_samples

```
Out[1]:
```
```python
# Compose a concise plain-English summary using the already-loaded summaries
# (variables metric_summary, trace_summary, log_summary, error_summary, error_samples exist in the kernel)

# defensive helpers
def safe_get_row(df, idx=0):
    if df is None or df.shape[0] == 0:
        return None
    try:
        return df.iloc[idx]
    except Exception:
        return None

top_metric = safe_get_row(metric_summary, 0)
top_trace = safe_get_row(trace_summary, 0)
top_log = safe_get_row(log_summary, 0)

# extract top 3 error rows
top_errors = []
if 'error_summary' in globals() and error_summary.shape[0] > 0:
    for _, r in error_summary.head(3).iterrows():
        top_errors.append((r['cmdb_id'], int(r['error_count']), r['earliest_ts'], r['latest_ts']))

# Build summary text
summary_lines = []
summary_lines.append("Summary of telemetry files (compact):")

if top_metric is not None:
    summary_lines.append(f"- metric.csv: top pair by count is ({top_metric['cmdb_id']}, {top_metric['kpi_name']}) "
                         f"with count={int(top_metric['count'])} and P95_value={float(top_metric['p95_value']):.4g}.")
else:
    summary_lines.append("- metric.csv: no data available.")

if top_trace is not None:
    summary_lines.append(f"- trace.csv: top pair by count is ({top_trace['cmdb_id']}, {top_trace['trace_name']}) "
                         f"with count={int(top_trace['count'])} and P95_value={float(top_trace['p95_value']):.4g}.")
else:
    summary_lines.append("- trace.csv: no data available.")

if top_log is not None:
    summary_lines.append(f"- log.csv: top pair by count is ({top_log['cmdb_id']}, {top_log['log_name']}) "
                         f"with count={int(top_log['count'])} and P95_value={float(top_log['p95_value']):.4g}.")
else:
    summary_lines.append("- log.csv: no data available.")

if len(top_errors) > 0:
    summary_lines.append("- error_logs.csv: error counts concentrated on frontend pods (top entries):")
    for cmdb, cnt, e_ts, l_ts in top_errors:
        summary_lines.append(f"  * {cmdb}: {cnt} errors, earliest={e_ts}, latest={l_ts}")
    # include sample message hints if available
    if 'error_samples' in globals() and error_samples.shape[0] > 0:
        # show up to 3 short samples
        samples = []
        for _, r in error_samples.head(3).iterrows():
            samples.append(f"{r['cmdb_id']} @ {r['ts_dt']}: {str(r['message'])[:80]}")  # truncate
        summary_lines.append("  Sample messages (truncated):")
        for s in samples:
            summary_lines.append(f"   - {s}")
else:
    summary_lines.append("- error_logs.csv: no error log entries found.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count     p95_value
4817  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30   1492.500000
0              adservice                                     app.grpc.count     30    225.550000
1              adservice                                       app.grpc.mrt     30      0.002366
2              adservice                                        app.grpc.rr     30    100.000000
3              adservice                                        app.grpc.sr     30     59.466477
4              adservice                                     app.http.count     30     24.000000
5              adservice                                       app.http.mrt     30      0.009296
6              adservice                                        app.http.rr     30    100.000000
7              adservice                                        app.http.sr     30    100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30   4990.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30   4990.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30      0.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose     30      0.000000
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...     30      1.000000
13             adservice  runtime.java_lang_Compilation_TotalCompilation...     30  42981.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30  32848.108333
15             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30      3.000000
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...     30  51656.079167
4778  shippingservice2-0  container.node-5.container_network_receive_MB....     30      0.037976
4779  shippingservice2-0  container.node-5.container_network_receive_err...     30      0.000000,                  cmdb_id                           trace_name  count  p95_value
1231  shippingservice2-0     trace.from_frontend2-0.row_count     30  15.550000
0            adservice-0  trace.from_frontend-0.duration_mean     30   0.000013
1            adservice-0   trace.from_frontend-0.duration_p95     30   0.000019
2            adservice-0     trace.from_frontend-0.error_rate     30   0.000000
3            adservice-0      trace.from_frontend-0.row_count     30  24.000000
4            adservice-0  trace.from_frontend-1.duration_mean     30   0.000015
5            adservice-0   trace.from_frontend-1.duration_p95     30   0.000028
6            adservice-0     trace.from_frontend-1.error_rate     30   0.000000
7            adservice-0      trace.from_frontend-1.row_count     30  19.000000
8            adservice-0  trace.from_frontend-2.duration_mean     30   0.000018
9            adservice-0   trace.from_frontend-2.duration_p95     30   0.000023
10           adservice-0     trace.from_frontend-2.error_rate     30   0.000000
11           adservice-0      trace.from_frontend-2.row_count     30  25.000000
12           adservice-1  trace.from_frontend-0.duration_mean     30   0.000014
13           adservice-1   trace.from_frontend-0.duration_p95     30   0.000023
14           adservice-1     trace.from_frontend-0.error_rate     30   0.000000
1192   shippingservice-1  trace.from_frontend-1.duration_mean     30   0.000067
1193   shippingservice-1   trace.from_frontend-1.duration_p95     30   0.000083
1194   shippingservice-1     trace.from_frontend-1.error_rate     30   0.000000
1195   shippingservice-1      trace.from_frontend-1.row_count     30   7.000000,               cmdb_id         log_name  count  p95_value
0         adservice-0  log.error_count     30       0.00
1         adservice-0    log.row_count     30    1416.10
2         adservice-1  log.error_count     30       0.00
3         adservice-1    log.row_count     30     129.10
4         adservice-2  log.error_count     30       0.00
5         adservice-2    log.row_count     30     133.10
6       cartservice-0  log.error_count     30       0.00
7       cartservice-0    log.row_count     30    1015.90
8       cartservice-1  log.error_count     30       0.00
9       cartservice-1    log.row_count     30    1010.55
10      cartservice-2  log.error_count     30       0.00
11      cartservice-2    log.row_count     30    1011.30
12  checkoutservice-0  log.error_count     30       0.00
13  checkoutservice-0    log.row_count     30     100.65
14  checkoutservice-1  log.error_count     30       0.00
15  checkoutservice-1    log.row_count     30      86.65
16  checkoutservice-2  log.error_count     30       0.00
17  checkoutservice-2    log.row_count     30      86.65
18  currencyservice-0  log.error_count     30       0.00
19  currencyservice-0    log.row_count     30     951.65,       cmdb_id  error_count              earliest_ts                latest_ts
2  frontend-2          664  2022-03-20 10:30:00 CST  2022-03-20 10:59:57 CST
0  frontend-0          604  2022-03-20 10:30:00 CST  2022-03-20 10:59:59 CST
1  frontend-1          483  2022-03-20 10:30:00 CST  2022-03-20 10:59:59 CST,          cmdb_id                    ts_dt                                            message
1750  frontend-0  2022-03-20 10:59:59 CST  severity: warning, message: failed to retrieve...
1749  frontend-1  2022-03-20 10:59:59 CST  severity: warning, message: failed to retrieve...
1747  frontend-2  2022-03-20 10:59:57 CST  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric if available, else load
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")

# Ensure correct types
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using ALL rows
grp = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    global_p95=('value', lambda s: float(s.quantile(0.95)) if s.count() > 0 else np.nan),
    global_p05=('value', lambda s: float(s.quantile(0.05)) if s.count() > 0 else np.nan),
    total_count=('value', 'count')
)

# 2) Filter to the UTC+8 window (inclusive)
start = pd.Timestamp('2022-03-20 10:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] <= end)].copy()

# 3) Attach global thresholds and mark anomalies
df_window = df_window.merge(grp, on=['cmdb_id', 'kpi_name'], how='left')
df_window['direction'] = np.where(df_window['value'] > df_window['global_p95'], 'high',
                                  np.where(df_window['value'] < df_window['global_p05'], 'low', None))

# 4) Identify consecutive anomalous runs (per minute) within the window
df_anom = df_window[df_window['direction'].notna()].copy()
df_anom = df_anom.sort_values(['cmdb_id', 'kpi_name', 'direction', 'ts']).reset_index(drop=True)

# compute gap in seconds between consecutive anomalous rows within same series+direction
df_anom['gap_s'] = df_anom.groupby(['cmdb_id', 'kpi_name', 'direction'])['ts'].diff().dt.total_seconds().fillna(0)
# start a new run if gap > 60 seconds
df_anom['new_run'] = (df_anom['gap_s'] > 60).astype(int)
df_anom['run_id'] = df_anom.groupby(['cmdb_id', 'kpi_name', 'direction'])['new_run'].cumsum()

runs = df_anom.groupby(['cmdb_id', 'kpi_name', 'direction', 'run_id'], as_index=False).agg(
    run_start_ts=('ts', 'min'),
    run_end_ts=('ts', 'max'),
    run_length_minutes=('ts', 'count'),
    run_max_value=('value', 'max'),
    run_min_value=('value', 'min'),
    global_p95=('global_p95', 'first'),
    global_p05=('global_p05', 'first')
)

# keep at most top 3 longest runs per (cmdb_id,kpi_name,direction)
runs = runs.sort_values(['cmdb_id', 'kpi_name', 'direction', 'run_length_minutes'], ascending=[True, True, True, False])
runs['rank_within_series'] = runs.groupby(['cmdb_id', 'kpi_name', 'direction'])['run_length_minutes'].rank(method='first', ascending=False)
runs_top3 = runs[runs['rank_within_series'] <= 3].copy()

# 5) Compute severity and sort across all series
runs_top3['severity'] = np.where(
    runs_top3['direction'] == 'high',
    runs_top3['run_max_value'] / runs_top3['global_p95'].replace({0: np.nan}),
    runs_top3['global_p05'].replace({0: np.nan}) / runs_top3['run_min_value'].replace({0: np.nan})
)
runs_top3['severity'] = runs_top3['severity'].replace([np.inf, -np.inf], np.nan).fillna(-1)

# Format timestamps compactly
runs_top3['run_start_ts'] = runs_top3['run_start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
runs_top3['run_end_ts'] = runs_top3['run_end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

final_cols = ['cmdb_id', 'kpi_name', 'direction', 'run_start_ts', 'run_end_ts',
              'run_length_minutes', 'run_max_value', 'run_min_value', 'global_p95', 'global_p05', 'severity']

runs_top20 = runs_top3.sort_values(['run_length_minutes', 'severity'], ascending=[False, False]).head(20)[final_cols].reset_index(drop=True)

# Keep intermediates in kernel: grp, df_window, df_anom, runs, runs_top3
runs_top20

```
Out[2]:
```
```python
# Compose a concise plain-English summary of the anomalous runs found (uses runs_top20 from kernel)
if 'runs_top20' not in globals() or runs_top20 is None or runs_top20.shape[0] == 0:
    summary = "No anomalous runs found in the metric window 2022-03-20 10:30:00 to 11:00:00 (UTC+8)."
else:
    df = runs_top20.copy()
    total_runs = df.shape[0]
    # Most runs are short (minutes). Compute basic stats
    min_len = int(df['run_length_minutes'].min())
    max_len = int(df['run_length_minutes'].max())
    high_count = int((df['direction'] == 'high').sum())
    low_count = int((df['direction'] == 'low').sum())

    lines = [
        f"Anomaly summary for metric window 2022-03-20 10:30–11:00 (UTC+8):",
        f"- Found {total_runs} anomalous runs (showing top candidates).",
        f"- Run lengths observed: min {min_len} min, max {max_len} min. (Most top runs are {max_len} minutes long.)",
        f"- Directions: {high_count} high runs, {low_count} low runs.",
        "",
        "Top anomalous runs (by run length then severity):"
    ]

    # include top 5 rows for brevity
    for _, r in df.head(5).iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        direction = r['direction']
        length = int(r['run_length_minutes'])
        start = r['run_start_ts']
        end = r['run_end_ts']
        sev = float(r['severity'])
        # shorten kpi name if long
        kpi_short = (kpi[:80] + '...') if len(kpi) > 80 else kpi
        lines.append(f"  - {cmdb} | {kpi_short} | {direction} | {length} min | {start} to {end} | severity={sev:.2f}")

    lines.append("")
    lines.append("Notes:")
    lines.append("- Severity calculated as run_max/global_p95 for high runs, or global_p05/run_min for low runs.")
    lines.append("- The top run by severity is emailservice-1 (high network receive) with severity ~17x above global P95.")
    lines.append("- Multiple node- and container-level KPIs show short (2-minute) anomalous bursts across services and nodes.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name direction             run_start_ts               run_end_ts  run_length_minutes  run_max_value  run_min_value  global_p95  global_p05   severity
0             emailservice-1  container.node-5.container_network_receive_MB....      high  2022-03-20 10:55:00 CST  2022-03-20 10:56:00 CST                   2       0.553773       0.032842    0.032138    0.021961  17.231294
1                     node-3                                     system.io.util      high  2022-03-20 10:35:00 CST  2022-03-20 10:36:00 CST                   2       3.050000       0.250000    0.227500    0.000000  13.406593
2          paymentservice2-0  container.node-6.container_network_receive_MB....      high  2022-03-20 10:32:00 CST  2022-03-20 10:33:00 CST                   2       0.378536       0.196448    0.119793    0.014805   3.159909
3          shippingservice-0  container.node-5.container_memory_failures.con...       low  2022-03-20 10:52:00 CST  2022-03-20 10:53:00 CST                   2       7.500000       3.000000   69.525000    9.075000   3.025000
4          shippingservice-0  container.node-5.container_memory_failures.hie...       low  2022-03-20 10:52:00 CST  2022-03-20 10:53:00 CST                   2       7.500000       3.000000   69.525000    9.075000   3.025000
5                adservice-1  container.node-5.container_memory_failures.con...      high  2022-03-20 10:33:00 CST  2022-03-20 10:34:00 CST                   2     116.500000      40.000000   38.650000    8.000000   3.014230
6                adservice-1  container.node-5.container_memory_failures.hie...      high  2022-03-20 10:33:00 CST  2022-03-20 10:34:00 CST                   2     116.500000      40.000000   38.650000    8.000000   3.014230
7                     node-2                                 system.io.avg_q_sz      high  2022-03-20 10:32:00 CST  2022-03-20 10:33:00 CST                   2       0.110000       0.070000    0.043000    0.000000   2.558140
8                     node-1                                      system.io.w_s       low  2022-03-20 10:30:00 CST  2022-03-20 10:31:00 CST                   2       5.940000       3.000000   67.100000    7.092000   2.364000
9   recommendationservice2-0  container.node-6.container_network_receive_MB....      high  2022-03-20 10:30:00 CST  2022-03-20 10:31:00 CST                   2       0.468251       0.302787    0.222516    0.080310   2.104345
10                    node-5                                      system.load.5       low  2022-03-20 10:30:00 CST  2022-03-20 10:31:00 CST                   2       0.580000       0.330000    2.152500    0.652000   1.975758
11                    node-6                                    system.io.svctm       low  2022-03-20 10:56:00 CST  2022-03-20 10:57:00 CST                   2       0.040000       0.030000    0.195500    0.058000   1.933333
12                    node-6                                     system.io.util       low  2022-03-20 10:56:00 CST  2022-03-20 10:57:00 CST                   2       0.050000       0.050000    1.077500    0.095000   1.900000
13   productcatalogservice-2  container.node-5.container_cpu_cfs_throttled_s...      high  2022-03-20 10:44:00 CST  2022-03-20 10:45:00 CST                   2       0.005712       0.005712    0.003142    0.000000   1.818182
14   productcatalogservice-0  container.node-5.container_cpu_cfs_throttled_s...      high  2022-03-20 10:51:00 CST  2022-03-20 10:52:00 CST                   2       0.003502       0.003502    0.001926    0.000000   1.818182
15   productcatalogservice-2  container.node-5.container_cpu_cfs_throttled_p...      high  2022-03-20 10:44:00 CST  2022-03-20 10:45:00 CST                   2       0.500000       0.500000    0.275000    0.000000   1.818182
16   productcatalogservice-2  mesh.destination.recommendationservice2.produc...      high  2022-03-20 10:33:00 CST  2022-03-20 10:34:00 CST                   2       0.000525       0.000525    0.000289    0.000000   1.818182
17                frontend-0  container.node-5.container_cpu_cfs_throttled_s...      high  2022-03-20 10:38:00 CST  2022-03-20 10:39:00 CST                   2       0.000949       0.000949    0.000522    0.000000   1.818182
18            cartservice2-0  container.node-6.container_cpu_cfs_throttled_s...      high  2022-03-20 10:38:00 CST  2022-03-20 10:39:00 CST                   2       0.152834       0.152834    0.084059    0.000000   1.818182
19               adservice-0  container.node-5.container_memory_failures.con...      high  2022-03-20 10:34:00 CST  2022-03-20 10:35:00 CST                   2     162.000000      91.000000   90.775000   50.950000   1.784632```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse runs_top20 from kernel
df = runs_top20.copy()

# Parse the displayed timestamp strings to real datetimes for correct sorting.
# They are formatted like '2022-03-20 10:55:00 CST' -> remove the ' CST' and localize to Asia/Shanghai
df['run_start_dt'] = pd.to_datetime(df['run_start_ts'].str.replace(r'\sCST$', '', regex=True),
                                    format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.tz_localize(tz)
df['run_end_dt'] = pd.to_datetime(df['run_end_ts'].str.replace(r'\sCST$', '', regex=True),
                                  format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.tz_localize(tz)

# Ensure severity numeric
df['severity'] = pd.to_numeric(df['severity'], errors='coerce').fillna(-1)

# Group by the original run_start_ts string (as requested), but use parsed dt for sorting
def agg_group(g):
    count = int(g.shape[0])
    earliest_end = g['run_end_dt'].min().strftime('%Y-%m-%d %H:%M:%S %Z')
    latest_end = g['run_end_dt'].max().strftime('%Y-%m-%d %H:%M:%S %Z')
    max_sev = float(g['severity'].max())
    # prefer highest-severity cmdb_ids: sort by severity desc, pick distinct up to 6
    cmdbs = g.sort_values('severity', ascending=False)['cmdb_id'].astype(str)
    cmdbs = list(dict.fromkeys(cmdbs))[:6]  # preserve order and unique
    cmdb_list = ",".join(cmdbs)
    return pd.Series({
        'run_start_ts': g['run_start_ts'].iat[0],
        'count_of_runs_starting_at_that_ts': count,
        'earliest_run_end_ts': earliest_end,
        'latest_run_end_ts': latest_end,
        'max_severity': max_sev,
        'cmdb_ids_top': cmdb_list,
        'run_start_dt_sort': g['run_start_dt'].iat[0]  # helper for sorting
    })

agg = df.groupby('run_start_ts', sort=False).apply(agg_group).reset_index(drop=True)

# Sort by the parsed datetime ascending
agg = agg.sort_values('run_start_dt_sort', ascending=True).drop(columns=['run_start_dt_sort']).reset_index(drop=True)

# Limit to at most 20 rows (there should be no more than unique start times in runs_top20)
agg_summary = agg.head(20)

agg_summary

```
Out[3]:
```
Aggregated anomalous runs by start time (timestamps are UTC+8):

- 2022-03-20 10:30:00 CST — 3 runs; end range 10:31:00; max severity 2.36; cmdbs: node-1, recommendationservice2-0, node-5.
- 2022-03-20 10:32:00 CST — 2 runs; end 10:33:00; max severity 3.16; cmdbs: paymentservice2-0, node-2.
- 2022-03-20 10:33:00 CST — 3 runs; end 10:34:00; max severity 3.01; cmdbs: adservice-1, productcatalogservice-2.
- 2022-03-20 10:34:00 CST — 1 run; end 10:35:00; max severity 1.78; cmdb: adservice-0.
- 2022-03-20 10:35:00 CST — 1 run; end 10:36:00; max severity 13.41; cmdb: node-3.
- 2022-03-20 10:38:00 CST — 2 runs; end 10:39:00; max severity 1.82; cmdbs: frontend-0, cartservice2-0.
- 2022-03-20 10:44:00 CST — 2 runs; end 10:45:00; max severity 1.82; cmdb: productcatalogservice-2.
- 2022-03-20 10:51:00 CST — 1 run; end 10:52:00; max severity 1.82; cmdb: productcatalogservice-0.
- 2022-03-20 10:52:00 CST — 2 runs; end 10:53:00; max severity 3.03; cmdb: shippingservice-0.
- 2022-03-20 10:55:00 CST — 1 run; end 10:56:00; max severity 17.23; cmdb: emailservice-1.
- 2022-03-20 10:56:00 CST — 2 runs; end 10:57:00; max severity 1.93; cmdb: node-6.

Key takeaways:
- Anomalies occur throughout the 10:30–10:57 window, often as short (2-minute) bursts.
- Highest severities: emailservice-1 at 10:55 (≈17x above global P95) and node-3 at 10:35 (≈13x).
- Clusters of multiple runs start at 10:30, 10:33 and 10:52, involving both nodes and services.

The original code execution output of IPython Kernel is also provided below for reference:

run_start_ts  count_of_runs_starting_at_that_ts      earliest_run_end_ts        latest_run_end_ts  max_severity                            cmdb_ids_top
0   2022-03-20 10:30:00 CST                                  3  2022-03-20 10:31:00 CST  2022-03-20 10:31:00 CST      2.364000  node-1,recommendationservice2-0,node-5
1   2022-03-20 10:32:00 CST                                  2  2022-03-20 10:33:00 CST  2022-03-20 10:33:00 CST      3.159909                paymentservice2-0,node-2
2   2022-03-20 10:33:00 CST                                  3  2022-03-20 10:34:00 CST  2022-03-20 10:34:00 CST      3.014230     adservice-1,productcatalogservice-2
3   2022-03-20 10:34:00 CST                                  1  2022-03-20 10:35:00 CST  2022-03-20 10:35:00 CST      1.784632                             adservice-0
4   2022-03-20 10:35:00 CST                                  1  2022-03-20 10:36:00 CST  2022-03-20 10:36:00 CST     13.406593                                  node-3
5   2022-03-20 10:38:00 CST                                  2  2022-03-20 10:39:00 CST  2022-03-20 10:39:00 CST      1.818182               frontend-0,cartservice2-0
6   2022-03-20 10:44:00 CST                                  2  2022-03-20 10:45:00 CST  2022-03-20 10:45:00 CST      1.818182                 productcatalogservice-2
7   2022-03-20 10:51:00 CST                                  1  2022-03-20 10:52:00 CST  2022-03-20 10:52:00 CST      1.818182                 productcatalogservice-0
8   2022-03-20 10:52:00 CST                                  2  2022-03-20 10:53:00 CST  2022-03-20 10:53:00 CST      3.025000                       shippingservice-0
9   2022-03-20 10:55:00 CST                                  1  2022-03-20 10:56:00 CST  2022-03-20 10:56:00 CST     17.231294                          emailservice-1
10  2022-03-20 10:56:00 CST                                  2  2022-03-20 10:57:00 CST  2022-03-20 10:57:00 CST      1.933333                                  node-6```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace if present, else load
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")

# ensure numeric and timestamp conversion
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Filter to the UTC+8 window inclusive
start = pd.Timestamp('2022-03-20 10:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 10:36:00', tz=tz)
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# 2) Group by (cmdb_id, trace_name) and aggregate
agg = df_trace_window.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count=('value', 'count'),
    mean_value=('value', 'mean'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    earliest_ts=('ts', 'min')
)

# format earliest_ts as compact UTC+8 string
agg['earliest_ts'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 3) Sort by mean_value desc and keep top 20
agg_top20 = agg.sort_values('mean_value', ascending=False).head(20).reset_index(drop=True)

# display the compact aggregation table
agg_top20[['cmdb_id', 'trace_name', 'count', 'mean_value', 'min_value', 'max_value', 'earliest_ts']]

```
Out[4]:
```
summary = (
    "Trace summary for 2022-03-20 10:30:00–10:36:00 (UTC+8): "
    "Top groups (each has 7 points in the 7-minute window): "
    "1) frontend-2 | trace.self.row_count — mean=1069.0 (min=986, max=1138); "
    "2) frontend-0 | trace.self.row_count — mean=922.71 (min=795, max=1114); "
    "3) frontend-1 | trace.self.row_count — mean=779.14 (min=694, max=859). "
    "Other high-mean groups include frontend2-0 and productcatalogservice2-0 self/forwarded row_count series (means ~527 and ~241), "
    "followed by several frontend→productcatalogservice and productcatalogservice from-frontend series (means ~164–141). "
    "Earliest timestamp for these top groups is 2022-03-20 10:30:00 CST. "
    "Interpretation: traffic/row counts are concentrated on frontend pods and flowing to productcatalogservice during this interval."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  count   mean_value  min_value  max_value              earliest_ts
0                 frontend-2                         trace.self.row_count      7  1069.000000      986.0     1138.0  2022-03-20 10:30:00 CST
1                 frontend-0                         trace.self.row_count      7   922.714286      795.0     1114.0  2022-03-20 10:30:00 CST
2                 frontend-1                         trace.self.row_count      7   779.142857      694.0      859.0  2022-03-20 10:30:00 CST
3                frontend2-0                         trace.self.row_count      7   527.428571      480.0      577.0  2022-03-20 10:30:00 CST
4                frontend2-0  trace.to_productcatalogservice2-0.row_count      7   240.714286      195.0      279.0  2022-03-20 10:30:00 CST
5   productcatalogservice2-0             trace.from_frontend2-0.row_count      7   240.714286      195.0      279.0  2022-03-20 10:30:00 CST
6                 frontend-2   trace.to_productcatalogservice-2.row_count      7   164.428571      148.0      184.0  2022-03-20 10:30:00 CST
7    productcatalogservice-2              trace.from_frontend-2.row_count      7   164.428571      148.0      184.0  2022-03-20 10:30:00 CST
8                 frontend-2   trace.to_productcatalogservice-0.row_count      7   164.428571      149.0      183.0  2022-03-20 10:30:00 CST
9                 frontend-2   trace.to_productcatalogservice-1.row_count      7   164.428571      149.0      184.0  2022-03-20 10:30:00 CST
10   productcatalogservice-0              trace.from_frontend-2.row_count      7   164.428571      149.0      183.0  2022-03-20 10:30:00 CST
11   productcatalogservice-1              trace.from_frontend-2.row_count      7   164.428571      149.0      184.0  2022-03-20 10:30:00 CST
12               frontend2-0        trace.to_currencyservice2-0.row_count      7   158.857143      110.0      204.0  2022-03-20 10:30:00 CST
13        currencyservice2-0             trace.from_frontend2-0.row_count      7   158.857143      110.0      204.0  2022-03-20 10:30:00 CST
14   productcatalogservice-1              trace.from_frontend-0.row_count      7   141.428571      123.0      166.0  2022-03-20 10:30:00 CST
15   productcatalogservice-0              trace.from_frontend-0.row_count      7   141.428571      124.0      166.0  2022-03-20 10:30:00 CST
16                frontend-0   trace.to_productcatalogservice-1.row_count      7   141.428571      123.0      166.0  2022-03-20 10:30:00 CST
17                frontend-0   trace.to_productcatalogservice-2.row_count      7   141.428571      123.0      167.0  2022-03-20 10:30:00 CST
18                frontend-0   trace.to_productcatalogservice-0.row_count      7   141.428571      124.0      166.0  2022-03-20 10:30:00 CST
19   productcatalogservice-2              trace.from_frontend-0.row_count      7   141.428571      123.0      167.0  2022-03-20 10:30:00 CST```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace if available, else load
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")

# ensure numeric and timestamp conversion (reuse existing columns if already present)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'ts' not in df_trace.columns or not pd.api.types.is_datetime64_any_dtype(df_trace['ts']):
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz is Asia/Shanghai
    df_trace['ts'] = pd.to_datetime(df_trace['ts']).dt.tz_convert(tz)

# Filter window inclusive (UTC+8)
start = pd.Timestamp('2022-03-20 10:30:00', tz=tz)
end = pd.Timestamp('2022-03-20 10:36:00', tz=tz)
df_trace_window2 = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# Select trace_name containing 'error_rate' or 'duration' (case-sensitive substring match)
mask = df_trace_window2['trace_name'].str.contains('error_rate|duration', regex=True)
df_trace_sel = df_trace_window2[mask].copy()

# Group and aggregate
agg = df_trace_sel.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count=('value', 'count'),
    mean_value=('value', 'mean'),
    max_value=('value', 'max'),
    earliest_ts=('ts', 'min')
)

# Format earliest_ts compactly (UTC+8)
agg['earliest_ts'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Sort by mean_value desc and keep top 20, select requested columns
agg_top20 = agg.sort_values('mean_value', ascending=False).head(20)[
    ['cmdb_id', 'trace_name', 'count', 'mean_value', 'max_value', 'earliest_ts']
].reset_index(drop=True)

# Display aggregated top groups only
agg_top20

```
Out[5]:
```
Summary (trace metrics for 2022-03-20 10:30–10:36, UTC+8; only trace names containing "error_rate" or "duration"):

- Top group by mean duration: frontend2-0 | trace.from_root.duration_p95 — mean 0.0626 s, max 0.0741 s, 7 points (earliest 2022-03-20 10:30:00 CST).
- Next highest: frontend-1 | trace.from_root.duration_p95 — mean 0.05915 s, max 0.06698 s, 7 points.
- frontend-0 | trace.from_root.duration_p95 — mean 0.05824 s, max 0.06194 s, 7 points.
- checkoutservice-0 related spans appear high in the top list (e.g., trace.from_frontend-0.duration_p95 and corresponding frontend→checkoutservice traces) with mean ≈0.055–0.058 s and some maxes up to ~0.101 s (≈101 ms).
- Most top groups have 7 points (one per minute across the 7-minute window); a few have 6 points.
- Earliest timestamp for these groups is 2022-03-20 10:30:00 CST for all listed top groups.
- Interpretation: during this interval, p95/duration signals are elevated mainly on frontend pods (from_root.duration_p95) and on checkout-related call paths, with observed p95 values up to ~0.10 s.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name  count  mean_value  max_value              earliest_ts
0          frontend2-0              trace.from_root.duration_p95      7    0.062591   0.074137  2022-03-20 10:30:00 CST
1           frontend-1              trace.from_root.duration_p95      7    0.059154   0.066978  2022-03-20 10:30:00 CST
2           frontend-0              trace.from_root.duration_p95      7    0.058238   0.061943  2022-03-20 10:30:00 CST
3    checkoutservice-0        trace.from_frontend-0.duration_p95      7    0.058062   0.091417  2022-03-20 10:30:00 CST
4           frontend-0   trace.to_checkoutservice-0.duration_p95      7    0.058062   0.091417  2022-03-20 10:30:00 CST
5           frontend-2              trace.from_root.duration_p95      7    0.057491   0.063344  2022-03-20 10:30:00 CST
6    checkoutservice-0       trace.from_frontend-0.duration_mean      7    0.055019   0.091417  2022-03-20 10:30:00 CST
7           frontend-0  trace.to_checkoutservice-0.duration_mean      7    0.055019   0.091417  2022-03-20 10:30:00 CST
8           frontend-0   trace.to_checkoutservice-2.duration_p95      7    0.051957   0.095497  2022-03-20 10:30:00 CST
9    checkoutservice-2        trace.from_frontend-0.duration_p95      7    0.051957   0.095497  2022-03-20 10:30:00 CST
10   checkoutservice-0        trace.from_frontend-1.duration_p95      6    0.048349   0.101199  2022-03-20 10:30:00 CST
11          frontend-1   trace.to_checkoutservice-0.duration_p95      6    0.048349   0.101199  2022-03-20 10:30:00 CST
12          frontend-1  trace.to_checkoutservice-0.duration_mean      6    0.047326   0.101199  2022-03-20 10:30:00 CST
13   checkoutservice-0       trace.from_frontend-1.duration_mean      6    0.047326   0.101199  2022-03-20 10:30:00 CST
14  checkoutservice2-0       trace.from_frontend2-0.duration_p95      6    0.045726   0.071085  2022-03-20 10:30:00 CST
15         frontend2-0  trace.to_checkoutservice2-0.duration_p95      6    0.045726   0.071085  2022-03-20 10:30:00 CST
16          frontend-0   trace.to_checkoutservice-1.duration_p95      7    0.044780   0.097191  2022-03-20 10:30:00 CST
17   checkoutservice-1        trace.from_frontend-0.duration_p95      7    0.044780   0.097191  2022-03-20 10:30:00 CST
18   checkoutservice-2       trace.from_frontend-0.duration_mean      7    0.044280   0.068474  2022-03-20 10:30:00 CST
19          frontend-0  trace.to_checkoutservice-2.duration_mean      7    0.044280   0.068474  2022-03-20 10:30:00 CST```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# reuse timezone and existing loaded variables where possible
tz = pytz.timezone('Asia/Shanghai')

# ensure df_error (error_logs) is loaded
if 'df_error' not in globals():
    df_error = pd.read_csv("error_logs.csv")

# normalize timestamps to timezone-aware datetimes (assume epoch seconds UTC then convert to Asia/Shanghai)
df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define windows (inclusive)
windows = {
    'A': (pd.Timestamp('2022-03-20 10:30:00', tz=tz), pd.Timestamp('2022-03-20 10:36:00', tz=tz)),
    'B': (pd.Timestamp('2022-03-20 10:50:00', tz=tz), pd.Timestamp('2022-03-20 11:00:00', tz=tz))
}

# 1) error_logs: aggregate per window and cmdb_id (top 10 per window)
error_agg_list = []
error_samples_list = []

for label, (start, end) in windows.items():
    df_w = df_error[(df_error['ts'] >= start) & (df_error['ts'] <= end)].copy()
    if df_w.shape[0] == 0:
        continue
    agg = df_w.groupby('cmdb_id', as_index=False).agg(
        error_count=('message', 'count'),
        earliest_ts=('ts', 'min'),
        latest_ts=('ts', 'max')
    )
    # format timestamps
    agg['earliest_ts'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['latest_ts'] = agg['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg['window_label'] = label
    # top 10 by error_count
    top10 = agg.sort_values('error_count', ascending=False).head(10)
    # reorder columns
    top10 = top10[['window_label', 'cmdb_id', 'error_count', 'earliest_ts', 'latest_ts']]
    error_agg_list.append(top10)

    # samples: up to 3 messages per cmdb_id for the cmdb_ids in top10 (choose latest messages)
    top_cmdbs = top10['cmdb_id'].tolist()
    df_samples = df_w[df_w['cmdb_id'].isin(top_cmdbs)].sort_values(['cmdb_id', 'ts'], ascending=[True, False])
    # pick up to 3 per cmdb_id
    df_samples = df_samples.groupby('cmdb_id').head(3).reset_index(drop=True)
    df_samples = df_samples[['cmdb_id', 'ts', 'message']].copy()
    df_samples['window_label'] = label
    df_samples['sample_ts'] = df_samples['ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # truncate message to 200 chars
    df_samples['sample_message'] = df_samples['message'].astype(str).str.slice(0, 200)
    df_samples = df_samples[['window_label', 'cmdb_id', 'sample_ts', 'sample_message']]
    error_samples_list.append(df_samples)

# combine per-window aggregates and samples
if len(error_agg_list) > 0:
    error_top_by_window = pd.concat(error_agg_list, ignore_index=True)
else:
    error_top_by_window = pd.DataFrame(columns=['window_label', 'cmdb_id', 'error_count', 'earliest_ts', 'latest_ts'])

if len(error_samples_list) > 0:
    error_samples_df = pd.concat(error_samples_list, ignore_index=True)
    # limit total sample rows to 30 as requested (preserve ordering by window_label then cmdb_id)
    error_samples_df = error_samples_df.head(30)
else:
    error_samples_df = pd.DataFrame(columns=['window_label', 'cmdb_id', 'sample_ts', 'sample_message'])

# 2) log.csv: filter for windows and log_name == 'log.error_count'
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# ensure timestamp tz-aware
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# assign window_label for rows falling into either window
def assign_window_label(ts):
    for lbl, (s, e) in windows.items():
        if (ts >= s) and (ts <= e):
            return lbl
    return None

df_log['window_label'] = df_log['ts'].apply(assign_window_label)
# filter for desired window rows and log_name
df_log_w = df_log[(df_log['window_label'].notna()) & (df_log['log_name'] == 'log.error_count')].copy()

# group by window_label and cmdb_id
if df_log_w.shape[0] > 0:
    log_agg = df_log_w.groupby(['window_label', 'cmdb_id'], as_index=False).agg(
        count=('value', 'count'),
        total_error_count=('value', 'sum'),
        mean_value=('value', 'mean'),
        p95_value=('value', lambda s: float(s.quantile(0.95)) if s.count() > 0 else np.nan)
    )
    log_agg_top20 = log_agg.sort_values('total_error_count', ascending=False).head(20)
else:
    log_agg_top20 = pd.DataFrame(columns=['window_label', 'cmdb_id', 'count', 'total_error_count', 'mean_value', 'p95_value'])

# Final compact outputs (store intermediates in variables above). Display aggregated results only:
error_top_by_window, error_samples_df, log_agg_top20

```
Out[6]:
```
Findings from targeted log correlation (UTC+8 windows A = 10:30–10:36, B = 10:50–11:00):

1) error_logs.csv (raw error messages)
- Window A (10:30–10:36): top cmdb_ids by message count
  - frontend-2: 128 messages (earliest 2022-03-20 10:30:00, latest 10:35:59)
  - frontend-0: 107 messages (earliest 10:30:00, latest 10:35:58)
  - frontend-1: 97 messages  (earliest 10:30:00, latest 10:35:58)
- Window B (10:50–11:00): top cmdb_ids by message count
  - frontend-2: 233 messages (earliest 10:50:00, latest 10:59:57)
  - frontend-0: 204 messages (earliest 10:50:00, latest 10:59:59)
  - frontend-1: 175 messages (earliest 10:50:10, latest 10:59:59)
- Sample messages (truncated) show the same recurring pattern across frontends in both windows, e.g.:
  - "severity: warning, message: failed to retrieve..." (multiple samples at ~10:35:5x in window A and ~10:59:5x in window B)

2) log.csv (log.error_count numeric metric)
- Window B (10 points per frontend):
  - frontend-2: count=10, total_error_count=233, mean=23.30, p95=25.0
  - frontend-0: count=10, total_error_count=204, mean=20.40, p95=23.1
  - frontend-1: count=10, total_error_count=175, mean=17.50, p95=20.1
- Window A (7 points per frontend):
  - frontend-2: count=7, total_error_count=152, mean=21.71, p95=24.0
  - frontend-0: count=7, total_error_count=129, mean=18.43, p95=21.7
  - frontend-1: count=7, total_error_count=108, mean=15.43, p95=17.7
- Many backend services showed zero log.error_count in these windows.

Concise conclusion / correlation
- Both raw error messages and the numeric log.error_count point to the frontend pods (frontend-2, frontend-0, frontend-1) as the primary sources of errors in both windows.
- Window B (10:50–11:00) is worse than window A — higher total error counts and higher per-minute means (peaks ~23 errors/min on frontend-2).
- Sample messages are consistent across frontends and windows ("failed to retrieve..."), indicating a common frontend-side failure or an upstream dependency causing repeated retrieval failures.
- Actionable next steps: investigate frontend service logs and the upstream dependency referenced by "failed to retrieve..." (network/backend dependency, cache, or downstream API) and correlate with traces/metrics (request latencies, downstream service errors) around the listed top timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(  window_label     cmdb_id  error_count              earliest_ts                latest_ts
0            A  frontend-2          128  2022-03-20 10:30:00 CST  2022-03-20 10:35:59 CST
1            A  frontend-0          107  2022-03-20 10:30:00 CST  2022-03-20 10:35:58 CST
2            A  frontend-1           97  2022-03-20 10:30:00 CST  2022-03-20 10:35:58 CST
3            B  frontend-2          233  2022-03-20 10:50:00 CST  2022-03-20 10:59:57 CST
4            B  frontend-0          204  2022-03-20 10:50:00 CST  2022-03-20 10:59:59 CST
5            B  frontend-1          175  2022-03-20 10:50:10 CST  2022-03-20 10:59:59 CST,    window_label     cmdb_id                sample_ts                                     sample_message
0             A  frontend-0  2022-03-20 10:35:58 CST  severity: warning, message: failed to retrieve...
1             A  frontend-0  2022-03-20 10:35:54 CST  severity: warning, message: failed to retrieve...
2             A  frontend-0  2022-03-20 10:35:52 CST  severity: warning, message: failed to retrieve...
3             A  frontend-1  2022-03-20 10:35:58 CST  severity: warning, message: failed to retrieve...
4             A  frontend-1  2022-03-20 10:35:54 CST  severity: warning, message: failed to retrieve...
5             A  frontend-1  2022-03-20 10:35:46 CST  severity: warning, message: failed to retrieve...
6             A  frontend-2  2022-03-20 10:35:59 CST  severity: warning, message: failed to retrieve...
7             A  frontend-2  2022-03-20 10:35:55 CST  severity: warning, message: failed to retrieve...
8             A  frontend-2  2022-03-20 10:35:51 CST  severity: warning, message: failed to retrieve...
9             B  frontend-0  2022-03-20 10:59:59 CST  severity: warning, message: failed to retrieve...
10            B  frontend-0  2022-03-20 10:59:53 CST  severity: warning, message: failed to retrieve...
11            B  frontend-0  2022-03-20 10:59:50 CST  severity: warning, message: failed to retrieve...
12            B  frontend-1  2022-03-20 10:59:59 CST  severity: warning, message: failed to retrieve...
13            B  frontend-1  2022-03-20 10:59:57 CST  severity: warning, message: failed to retrieve...
14            B  frontend-1  2022-03-20 10:59:54 CST  severity: warning, message: failed to retrieve...
15            B  frontend-2  2022-03-20 10:59:57 CST  severity: warning, message: failed to retrieve...
16            B  frontend-2  2022-03-20 10:59:52 CST  severity: warning, message: failed to retrieve...
17            B  frontend-2  2022-03-20 10:59:51 CST  severity: warning, message: failed to retrieve...,    window_label            cmdb_id  count  total_error_count  mean_value  p95_value
48            B         frontend-2     10              233.0   23.300000       25.0
46            B         frontend-0     10              204.0   20.400000       23.1
47            B         frontend-1     10              175.0   17.500000       20.1
17            A         frontend-2      7              152.0   21.714286       24.0
15            A         frontend-0      7              129.0   18.428571       21.7
16            A         frontend-1      7              108.0   15.428571       17.7
6             A  checkoutservice-0      7                0.0    0.000000        0.0
7             A  checkoutservice-1      7                0.0    0.000000        0.0
8             A  checkoutservice-2      7                0.0    0.000000        0.0
9             A  currencyservice-0      7                0.0    0.000000        0.0
10            A  currencyservice-1      7                0.0    0.000000        0.0
2             A        adservice-2      7                0.0    0.000000        0.0
0             A        adservice-0      7                0.0    0.000000        0.0
1             A        adservice-1      7                0.0    0.000000        0.0
13            A     emailservice-1      7                0.0    0.000000        0.0
12            A     emailservice-0      7                0.0    0.000000        0.0
11            A  currencyservice-2      7                0.0    0.000000        0.0
14            A     emailservice-2      7                0.0    0.000000        0.0
18            A   paymentservice-0      7                0.0    0.000000        0.0
19            A   paymentservice-1      7                0.0    0.000000        0.0)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace if already loaded, else load from CSV
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")

# ensure numeric 'value'
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# ensure timezone-aware timestamp column 'ts' in UTC+8
if 'ts' not in df_trace.columns or not pd.api.types.is_datetime64_any_dtype(df_trace['ts']):
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # coerce to datetime and convert tz to Asia/Shanghai
    df_trace['ts'] = pd.to_datetime(df_trace['ts']).dt.tz_convert(tz)

# Define window (inclusive) in UTC+8
start = pd.Timestamp('2022-03-20 10:50:00', tz=tz)
end = pd.Timestamp('2022-03-20 11:00:00', tz=tz)

# Filter to window
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# Select trace_name containing 'error_rate' or 'duration' (case-sensitive)
mask = df_trace_window['trace_name'].str.contains('error_rate|duration', regex=True, na=False)
df_trace_sel = df_trace_window[mask].copy()

# Group and aggregate
agg = df_trace_sel.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count=('value', 'count'),
    mean_value=('value', 'mean'),
    max_value=('value', 'max'),
    earliest_ts=('ts', 'min')
)

# Format earliest_ts compactly (UTC+8)
agg['earliest_ts'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Sort by mean_value desc and keep top 20, select requested columns
agg_top20 = agg.sort_values('mean_value', ascending=False).head(20)[
    ['cmdb_id', 'trace_name', 'count', 'mean_value', 'max_value', 'earliest_ts']
].reset_index(drop=True)

# Display the compact aggregated result
agg_top20

```
Out[7]:
```
```python
summary = (
    "Trace summary for 2022-03-20 10:50:00–11:00:00 (UTC+8): "
    "Top mean durations are on frontend pods: frontend2-0 trace.from_root.duration_p95 mean=0.05854s, "
    "frontend-2 mean=0.05789s, frontend-1 mean=0.05783s, frontend-0 mean=0.05748s (each with 10 samples). "
    "Checkout-related spans also appear elevated: e.g., checkoutservice-0 trace.from_frontend-2.duration_p95 mean=0.05065s (max up to ~0.096s) and matching frontend→checkoutservice traces. "
    "Most top groups have full coverage across the window (count=10); earliest timestamps are 2022-03-20 10:50:00 CST. "
    "Interpretation: during this interval frontends show modestly higher p95 durations (~55–60 ms) and the checkout call path exhibits higher p95 spikes (up to ~96 ms), indicating checkout-related latency impact."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name  count  mean_value  max_value              earliest_ts
0          frontend2-0              trace.from_root.duration_p95     10    0.058541   0.073621  2022-03-20 10:50:00 CST
1           frontend-2              trace.from_root.duration_p95     10    0.057889   0.068087  2022-03-20 10:50:00 CST
2           frontend-1              trace.from_root.duration_p95     10    0.057834   0.068274  2022-03-20 10:50:00 CST
3           frontend-0              trace.from_root.duration_p95     10    0.057479   0.063007  2022-03-20 10:50:00 CST
4    checkoutservice-0        trace.from_frontend-2.duration_p95     10    0.050647   0.096269  2022-03-20 10:50:00 CST
5           frontend-2   trace.to_checkoutservice-0.duration_p95     10    0.050647   0.096269  2022-03-20 10:50:00 CST
6           frontend-2  trace.to_checkoutservice-0.duration_mean     10    0.045667   0.086935  2022-03-20 10:50:00 CST
7    checkoutservice-0       trace.from_frontend-2.duration_mean     10    0.045667   0.086935  2022-03-20 10:50:00 CST
8           frontend-2   trace.to_checkoutservice-2.duration_p95     10    0.041145   0.082268  2022-03-20 10:50:00 CST
9    checkoutservice-2        trace.from_frontend-2.duration_p95     10    0.041145   0.082268  2022-03-20 10:50:00 CST
10         frontend2-0  trace.to_checkoutservice2-0.duration_p95      9    0.041120   0.047704  2022-03-20 10:50:00 CST
11  checkoutservice2-0       trace.from_frontend2-0.duration_p95      9    0.041120   0.047704  2022-03-20 10:50:00 CST
12         frontend2-0             trace.from_root.duration_mean     10    0.039931   0.044099  2022-03-20 10:50:00 CST
13   checkoutservice-1        trace.from_frontend-2.duration_p95     10    0.039660   0.075440  2022-03-20 10:50:00 CST
14          frontend-2   trace.to_checkoutservice-1.duration_p95     10    0.039660   0.075440  2022-03-20 10:50:00 CST
15          frontend-2             trace.from_root.duration_mean     10    0.039299   0.042732  2022-03-20 10:50:00 CST
16          frontend-1   trace.to_checkoutservice-2.duration_p95     10    0.039141   0.056294  2022-03-20 10:50:00 CST
17   checkoutservice-2        trace.from_frontend-1.duration_p95     10    0.039141   0.056294  2022-03-20 10:50:00 CST
18   checkoutservice-2       trace.from_frontend-1.duration_mean     10    0.038919   0.056294  2022-03-20 10:50:00 CST
19          frontend-1  trace.to_checkoutservice-2.duration_mean     10    0.038919   0.056294  2022-03-20 10:50:00 CST```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# reuse timezone and df_trace if available
tz = pytz.timezone('Asia/Shanghai')

if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")

# ensure numeric value and timezone-aware timestamp in Asia/Shanghai
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
if 'ts' not in df_trace.columns or not pd.api.types.is_datetime64_any_dtype(df_trace['ts']):
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    df_trace['ts'] = pd.to_datetime(df_trace['ts']).dt.tz_convert(tz)

# define window and target cmdb_ids
start = pd.Timestamp('2022-03-20 10:50:00', tz=tz)
end = pd.Timestamp('2022-03-20 11:00:00', tz=tz)
targets = ["emailservice-0","emailservice-1","emailservice-2",
           "shippingservice-0","shippingservice-1","shippingservice-2"]

# filter rows by time window and cmdb_id, and trace_name containing 'error_rate' or 'duration'
df_win = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end) & (df_trace['cmdb_id'].isin(targets))].copy()
mask = df_win['trace_name'].str.contains('error_rate|duration', regex=True, na=False)
df_sel = df_win[mask].copy()

# group and aggregate
agg = df_sel.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count=('value', 'count'),
    mean_value=('value', 'mean'),
    max_value=('value', 'max'),
    earliest_ts=('ts', 'min')
)

# format earliest_ts compactly (UTC+8)
agg['earliest_ts'] = agg['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# sort by mean_value desc and keep up to 20 rows, select requested columns
agg_top20 = agg.sort_values('mean_value', ascending=False).head(20)[
    ['cmdb_id', 'trace_name', 'count', 'mean_value', 'max_value', 'earliest_ts']
].reset_index(drop=True)

# display compact result
agg_top20

```
Out[8]:
```
```python
summary = (
    "Trace window 2022-03-20 10:50–11:00 (UTC+8) for selected services:\n"
    "- Email services (emailservice-0/1/2) show the highest mean durations among the selected set, "
    "about 0.00021–0.00022 seconds (≈0.21–0.22 ms), with maxima up to ~0.000275 s (0.275 ms). Counts per series are 8–10 (most are 10) and earliest samples are at 2022-03-20 10:50:00 CST.\n"
    "- Shippingservice (shippingservice-0/1) duration p95 means are much lower (~0.00008–0.00009 s).\n"
    "- The filtered traces were duration metrics (no notable error_rate entries in this subset).\n"
    "Interpretation: emailservice shows slightly higher checkout-related durations compared with shippingservice, but absolute latency values are extremely small (<1 ms). There is no evidence of a meaningful latency spike for these services in this interval."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  count  mean_value  max_value              earliest_ts
0      emailservice-1   trace.from_checkoutservice-1.duration_p95      9    0.000222   0.000237  2022-03-20 10:50:00 CST
1      emailservice-2   trace.from_checkoutservice-0.duration_p95     10    0.000219   0.000240  2022-03-20 10:50:00 CST
2      emailservice-1  trace.from_checkoutservice-1.duration_mean      9    0.000219   0.000237  2022-03-20 10:50:00 CST
3      emailservice-1   trace.from_checkoutservice-2.duration_p95     10    0.000219   0.000275  2022-03-20 10:50:00 CST
4      emailservice-0   trace.from_checkoutservice-0.duration_p95      8    0.000218   0.000232  2022-03-20 10:50:00 CST
5      emailservice-2  trace.from_checkoutservice-0.duration_mean     10    0.000216   0.000233  2022-03-20 10:50:00 CST
6      emailservice-1  trace.from_checkoutservice-2.duration_mean     10    0.000215   0.000243  2022-03-20 10:50:00 CST
7      emailservice-0  trace.from_checkoutservice-0.duration_mean      8    0.000215   0.000222  2022-03-20 10:50:00 CST
8      emailservice-0   trace.from_checkoutservice-2.duration_p95     10    0.000214   0.000227  2022-03-20 10:50:00 CST
9      emailservice-0   trace.from_checkoutservice-1.duration_p95     10    0.000214   0.000221  2022-03-20 10:50:00 CST
10     emailservice-2   trace.from_checkoutservice-2.duration_p95     10    0.000213   0.000245  2022-03-20 10:50:00 CST
11     emailservice-0  trace.from_checkoutservice-2.duration_mean     10    0.000213   0.000227  2022-03-20 10:50:00 CST
12     emailservice-0  trace.from_checkoutservice-1.duration_mean     10    0.000213   0.000221  2022-03-20 10:50:00 CST
13     emailservice-2   trace.from_checkoutservice-1.duration_p95     10    0.000212   0.000220  2022-03-20 10:50:00 CST
14     emailservice-1   trace.from_checkoutservice-0.duration_p95     10    0.000211   0.000221  2022-03-20 10:50:00 CST
15     emailservice-2  trace.from_checkoutservice-1.duration_mean     10    0.000211   0.000220  2022-03-20 10:50:00 CST
16     emailservice-2  trace.from_checkoutservice-2.duration_mean     10    0.000211   0.000228  2022-03-20 10:50:00 CST
17     emailservice-1  trace.from_checkoutservice-0.duration_mean     10    0.000210   0.000217  2022-03-20 10:50:00 CST
18  shippingservice-1          trace.from_frontend-2.duration_p95     10    0.000088   0.000213  2022-03-20 10:50:00 CST
19  shippingservice-0          trace.from_frontend-0.duration_p95     10    0.000084   0.000239  2022-03-20 10:50:00 CST```
```